# Line movement: poll the odds and chart how prices move

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/03-line-movement.ipynb)

Odds are not static: prices drift as money and news arrive, and the drift itself is
signal (steam moves, reverse line movement, closing line value). This notebook
polls the ParlayAPI odds feed on a fixed interval, records every price it sees, and
charts the movement per bookmaker with matplotlib.

Works with no key (demo endpoint, 60 requests per hour, so keep the poll count
modest). A [free key](https://parlay-api.com/signup) raises the ceiling and adds books and markets.

Honest expectation-setting: over the couple of minutes this demo polls, lines often
do not move at all, especially hours before a game. The plumbing is the point; run
it longer (raise `POLLS` and `INTERVAL_SECONDS`) or near game time to see real
movement. Keyed users can also pull server-side history from
`GET /v1/sports/{sport}/line-movement` instead of polling.

**Default run:** offline, with no API calls or key prompts. Existing mathematical examples
are illustrative calculations, not current quotes. Select a live mode explicitly in the next cell.


In [ ]:
# Run All is offline by default. Choose demo or account explicitly for API calls.
MODE = "offline"  # "offline", "demo", or "account"
SPORT = "baseball_mlb"
RUN_EXTRA_API_CHECKS = False
RUN_POLLING = False
SAVE_PRIVATE_CSV = False

import getpass
import json
import requests

SPORTS = {"baseball_mlb", "basketball_nba", "americanfootball_nfl",
          "icehockey_nhl", "soccer_epl", "mma_mixed_martial_arts"}
BASE_URL = "https://parlay-api.com"
_runtime_key = None

def get_runtime_key():
    global _runtime_key
    if MODE != "account":
        raise RuntimeError("Choose account mode before entering a key.")
    if _runtime_key is None:
        value = getpass.getpass("Your own ParlayAPI key (hidden; account credits apply): ")
        if not value or any(ord(c) < 33 or ord(c) > 126 for c in value):
            raise RuntimeError("Enter a valid key without whitespace or control characters.")
        _runtime_key = value
    return _runtime_key

def request_json(path, *, params=None, method="GET", body=None, account=False):
    if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
        raise RuntimeError("Choose a listed mode and supported sport.")
    if MODE == "offline":
        raise RuntimeError("Offline mode makes no API requests.")
    if not path.startswith("/") or path.startswith("//") or ".." in path or "\\" in path:
        raise RuntimeError("Use a fixed API path.")
    headers = {"Accept": "application/json"}
    if account:
        headers["X-API-Key"] = get_runtime_key()
    try:
        with requests.request(method, "https://parlay-api.com" + path,
                              params=params, json=body, headers=headers,
                              timeout=30, allow_redirects=False, stream=True) as response:
            if response.status_code != 200:
                raise RuntimeError(f"API returned HTTP {response.status_code}. No automatic retry.")
            chunks = []
            size = 0
            for chunk in response.iter_content(65536):
                size += len(chunk)
                if size > 10_000_000:
                    raise RuntimeError("Response exceeds the size limit.")
                chunks.append(chunk)
            return json.loads(b"".join(chunks))
    except (requests.RequestException, ValueError):
        raise RuntimeError("Request or JSON response failed. No automatic retry.") from None

if MODE not in {"offline", "demo", "account"} or SPORT not in SPORTS:
    raise RuntimeError("Choose a listed mode and supported sport.")
print("Mode:", MODE)
print("Offline runs the math without network. Demo is a limited anonymous sample.")
print("Account mode prompts at runtime and uses your own allowance. Keep that copy private.")


In [ ]:
def fetch_odds(sport=None, markets="h2h,spreads,totals", odds_format="american"):
    """A chosen demo or account request. Offline returns no live observations."""
    sport = sport or SPORT
    if sport not in SPORTS or odds_format != "american":
        raise RuntimeError("Choose a supported sport and American odds.")
    requested = markets.split(",")
    if not requested or any(m not in {"h2h", "spreads", "totals"} for m in requested):
        raise RuntimeError("Choose h2h, spreads, or totals.")
    if MODE == "offline":
        return []
    if MODE == "account":
        events = request_json(f"/v1/sports/{sport}/odds", account=True,
                              params={"markets": markets,
                                      "oddsFormat": "american"})
    else:
        payload = request_json(f"/v1/try/{sport}/odds")
        if (not isinstance(payload, dict) or payload.get("demo") is not True
                or not isinstance(payload.get("events"), list) or len(payload["events"]) > 5):
            raise RuntimeError("Unexpected demo response. No observations used.")
        events = payload["events"]
    if not isinstance(events, list) or any(not isinstance(e, dict) or e.get("sport_key") != sport for e in events):
        raise RuntimeError("Response does not match the chosen sport.")
    return events


## Poll on an interval

Each poll fetches the current board and appends one row per (bookmaker, outcome)
for the first event on the slate. Keep `POLLS * INTERVAL_SECONDS` small when
keyless: the demo endpoint allows 60 requests per hour.

In [ ]:
import time
import pandas as pd
from datetime import datetime, timezone

POLLS = 5
INTERVAL_SECONDS = 12

records = []
event_id = None
event_label = None

for poll in range(1, POLLS + 1) if RUN_POLLING and MODE != "offline" else []:
    ts = datetime.now(timezone.utc)
    try:
        events = fetch_odds(markets="h2h")
    except Exception:
        print(f"poll {poll}: fetch failed (request failed); skipping")
        events = []
    if events and event_id is None:
        event_id = events[0]["id"]
        event_label = f"{events[0]['away_team']} at {events[0]['home_team']}"
        print(f"Tracking: {event_label} (starts {events[0]['commence_time']})")
    target = next((e for e in events if e["id"] == event_id), None)
    if target is None:
        print(f"poll {poll}: tracked event not on the board")
    else:
        for bm in target.get("bookmakers", []):
            for mkt in bm.get("markets", []):
                if mkt["key"] != "h2h":
                    continue
                for out in mkt.get("outcomes", []):
                    records.append({"poll": poll, "time_utc": ts,
                                    "bookmaker": bm["key"], "outcome": out["name"],
                                    "price": out["price"]})
        print(f"poll {poll}/{POLLS} at {ts:%H:%M:%S}Z: "
              f"{len(target.get('bookmakers', []))} books recorded")
    if poll < POLLS:
        time.sleep(INTERVAL_SECONDS)

history = pd.DataFrame(records)
print(f"\ncollected {len(history)} price observations")

## The movement table

Before any chart, look at the raw numbers: first and last observed price per book
and side, and the delta in American cents. (A price that goes from -110 to -115
moved 5 cents against you.)

In [ ]:
if history.empty:
    print("No data collected; nothing to summarize.")
else:
    summary = (history.sort_values("poll")
               .groupby(["bookmaker", "outcome"])["price"]
               .agg(first_price="first", last_price="last", observations="count")
               .reset_index())
    summary["moved_cents"] = summary["last_price"] - summary["first_price"]
    display(summary)
    if (summary["moved_cents"] == 0).all():
        print("No movement during this window, which is common over a couple of")
        print("minutes. Raise POLLS / INTERVAL_SECONDS or run closer to game time.")

## Chart it

One panel per side of the market, one line per bookmaker, steps because a price
holds until it changes. Matplotlib only, default color cycle, single y axis. To
keep the chart readable the plot caps itself at the 6 most-observed books; raise
`BOOKS_TO_PLOT` to see them all (the summary table above already does).

In [ ]:
import matplotlib.pyplot as plt

if history.empty:
    print("Nothing to plot.")
else:
    BOOKS_TO_PLOT = 6
    top_books = (history.groupby("bookmaker")["price"].count()
                 .sort_values(ascending=False).head(BOOKS_TO_PLOT).index)
    plotted = history[history["bookmaker"].isin(top_books)]
    n_books = history["bookmaker"].nunique()
    if n_books > BOOKS_TO_PLOT:
        print(f"plotting {BOOKS_TO_PLOT} of {n_books} books; raise BOOKS_TO_PLOT for more")
    outcomes = list(plotted["outcome"].unique())
    fig, axes = plt.subplots(1, len(outcomes), figsize=(7 * len(outcomes), 4.5),
                             sharey=False, squeeze=False)
    for ax, outcome in zip(axes[0], outcomes):
        side = plotted[plotted["outcome"] == outcome]
        for book, grp in side.groupby("bookmaker"):
            grp = grp.sort_values("poll")
            ax.plot(grp["poll"], grp["price"], drawstyle="steps-post",
                    linewidth=2, marker="o", markersize=5, label=book)
        ax.set_title(outcome)
        ax.set_xlabel("poll number")
        ax.set_ylabel("American price")
        ax.set_xticks(sorted(side["poll"].unique()))
        ax.grid(True, alpha=0.25)
        ax.legend(title="bookmaker", fontsize=9)
    fig.suptitle(f"Moneyline by poll: {event_label}" if event_label else "Moneyline by poll")
    plt.tight_layout()
    plt.show()

## Where to take this

- Persist every poll to CSV or a database (notebook 01 has the flattener) and you
  are building your own line-movement archive.
- Compare your entry price to the last pre-game price you observed: that is
  closing line value, the subject of notebook 04.
- Keyed plans can skip client-side polling: `GET /v1/sports/{sport}/line-movement`
  serves movement history, and WebSocket / SSE streaming exists on the Business
  tier and up (see [parlay-api.com/pricing](https://parlay-api.com/pricing)).

---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for personal and internal research and education. Nothing here is betting advice.

## Private runtime data

Keep this notebook's code shareable and your account work private. Do not paste keys into
code cells, save them in notebook text, or commit downloaded observations. The hidden prompt
keeps the key in this runtime only. Clear all outputs before sharing or saving to GitHub;
Colab's output-omission setting is an additional safeguard, not a guarantee on other hosts.
The original published notebook contains no saved API results. Do not share an executed
account notebook or its exports. Each person uses their own account and key.

The MIT license covers code. API access does not grant public redisplay or redistribution
rights. Your applicable [terms](https://parlay-api.com/terms) and written agreement govern data.
Current coverage and plans: [docs](https://parlay-api.com/docs), [pricing](https://parlay-api.com/pricing).


In [ ]:
# Drop the runtime reference when finished. Restart the runtime to release other state.
_runtime_key = None
